<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/06-complex-nested-data/04-programmatic-flattening.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 6.4 — Programmatic flattening

Build the two utilities, run them on events.jsonl, then stress-test on a
5-level-deep synthetic payload the utilities have never seen.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, IntegerType,
    DoubleType, ArrayType,
)

spark = (
    SparkSession.builder
    .appName("6.4-flattening")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

EVENTS_SCHEMA = StructType([
    StructField("event_id", LongType(), False),
    StructField("ts",       StringType(), True),
    StructField("action",   StringType(), True),
    StructField("user", StructType([
        StructField("id",      StringType(), True),
        StructField("country", StringType(), True),
    ]), True),
    StructField("items", ArrayType(StructType([
        StructField("sku", StringType(),  True),
        StructField("qty", IntegerType(), True),
    ])), True),
    StructField("payment", StructType([
        StructField("method", StringType(), True),
        StructField("amount", DoubleType(), True),
    ]), True),
])
events = (
    spark.read.schema(EVENTS_SCHEMA)
    .option("mode", "DROPMALFORMED")
    .json(f"{DATA_DIR}/events.jsonl")
)

## Utility 1: flatten structs only (arrays left intact)

In [ ]:
def flatten_struct_cols(schema, prefix=""):
    """Select-expressions flattening all structs; arrays pass through."""
    cols = []
    for field in schema.fields:
        path = f"{prefix}.{field.name}" if prefix else field.name
        name = path.replace(".", "_")
        if isinstance(field.dataType, StructType):
            cols += flatten_struct_cols(field.dataType, prefix=path)
        else:
            cols.append(col(path).alias(name))
    return cols

flat = events.select(flatten_struct_cols(events.schema))
flat.printSchema()   # user_id, user_country, payment_method... items still an array
print("rows unchanged:", flat.count() == events.count())

## Utility 2: flatten EVERYTHING (explode_outer arrays, iterate to fixpoint)

In [ ]:
def flatten_all(df, sep="_"):
    """Fully rectangular output. Row count grows at each explode!"""
    while True:
        nested = next((f for f in df.schema.fields
                       if isinstance(f.dataType, (StructType, ArrayType))), None)
        if nested is None:
            return df
        c = nested.name
        if isinstance(nested.dataType, StructType):
            others = [n for n in df.columns if n != c]
            df = df.select(*others, *[
                col(f"{c}.{sub.name}").alias(f"{c}{sep}{sub.name}")
                for sub in nested.dataType.fields])
        else:
            df = df.withColumn(c, F.explode_outer(c))

rect = flatten_all(events)
rect.printSchema()
print(f"rows: {events.count()} events -> {rect.count()} rows (one per event x item; explode_outer kept item-less events)")
rect.select("event_id", "user_id", "items_sku", "items_qty", "payment_amount").show()

## Stress test: a payload these utilities have never seen

In [ ]:
gnarly = spark.read.json(spark.sparkContext.parallelize(['''
{"id": 1,
 "meta": {"source": {"system": "crm", "region": {"code": "eu-1", "zone": "a"}},
           "batch": 42},
 "customer": {"name": "Acme", "contacts": [
     {"kind": "email", "value": "a@acme.io", "prefs": {"newsletter": true}},
     {"kind": "phone", "value": "+49-123",   "prefs": {"newsletter": false}}]}}
''']))

gnarly.printSchema()
flatten_all(gnarly).show(truncate=False)   # 5 levels + array-of-structs-with-structs: handled

## The multiplicative-arrays hazard, demonstrated

In [ ]:
two_arrays = spark.createDataFrame(
    [(1, ["p1", "p2", "p3"], ["e1", "e2"])],
    "id INT, phones ARRAY<STRING>, emails ARRAY<STRING>",
)
print("1 input row ->", flatten_all(two_arrays).count(), "rows (3 x 2 cross product!)")
flatten_all(two_arrays).show()
# usually a modeling error: these arrays are INDEPENDENT — flatten one, or neither

## Schema drift as data: leaf-path diffing

In [ ]:
def leaf_paths(schema, prefix=""):
    paths = set()
    for f in schema.fields:
        p = f"{prefix}.{f.name}" if prefix else f.name
        if isinstance(f.dataType, StructType):
            paths |= leaf_paths(f.dataType, p)
        elif isinstance(f.dataType, ArrayType) and isinstance(f.dataType.elementType, StructType):
            paths |= leaf_paths(f.dataType.elementType, p + "[]")
        else:
            paths.add(p)
    return paths

contract = leaf_paths(EVENTS_SCHEMA)
observed = leaf_paths(spark.read.option("mode", "DROPMALFORMED")
                      .json(f"{DATA_DIR}/events.jsonl").schema)   # inference, exploration-only

print("in data but not contract:", observed - contract)
print("in contract but not data:", contract - observed)

## Exercises

1. Add a `max_rows_factor` guard to `flatten_all`: estimate growth before each explode (`avg(size(col))`) and raise if cumulative factor exceeds the limit. Test on `two_arrays`.
2. Add an `explode_allowlist` parameter: arrays NOT on the list are kept (or converted to JSON strings with `to_json` — the 6.3 boundary trick). Rectangle guaranteed, rows controlled.
3. `leaf_paths` marks array hops with `[]`. Extend the drift report to also flag TYPE changes for paths present in both schemas.
4. Boundary run: flatten `events` (structs-only version), write to CSV — does it succeed now? Try again with the fully-rectangular version. Which columns blocked the first attempt?

In [ ]:
# your exercise code here
